# IMCO 

Measures from "mapeando la corrupcion"

In [1]:
import pandas as pd
import numpy as np
from pyprojroot import here
import statsmodels.api as sm
import statsmodels.formula.api as smf
#from unidecode import unidecode
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import utils

processed_data = here('data/processed_data')


In [2]:
var2keep = [
    #identifiers
    'supplier_name_clean',
    #'contract_code',
    'purchasing_unit_id',
    #'file_code',

    #variables
    'contract_price_mx',
    #'award_date',
    #'contract_signature_date',
    'procedure_type_fixed',
    'contract_initial_date',
    'contract_year',
    'providers_registry_code']

    # #indepedent variables
    # 'call_for_tender_publication',
    # 'procedure_type',
    # 'procedure_type_fixed',
    # 'submission_period',
    # 'decision_period',
    # 'buyer_dependence',
    # 'bl_conformity',
    # 'contract_price_mx',

    # #controls
    # 'contract_year',    
    # 'contract_price_decile',
    # 'government_level',
    # 'supply_type',
    # 'state']

In [3]:
#read feather
contracts = pd.read_feather(processed_data / 'mxc11to22_base.feather', columns=var2keep)
contracts.shape

(2308908, 7)

In [4]:
contracts['prov_id'] = contracts.index

### Proportion of direct procedures

In [5]:
buyers = contracts[['contract_year', 'purchasing_unit_id', 'procedure_type_fixed']]
buyers = buyers.groupby(['contract_year', 'purchasing_unit_id', 'procedure_type_fixed']).size().reset_index(name='ncontracts_per_procedure')
buyers['ncontracts'] = buyers.groupby(['contract_year', 'purchasing_unit_id'])['ncontracts_per_procedure'].transform('sum')
buyers['prop_per_procedure'] = buyers['ncontracts_per_procedure'] / buyers['ncontracts']
buyers

,contract_year,purchasing_unit_id,procedure_type_fixed,ncontracts_per_procedure,ncontracts,prop_per_procedure
0,2011,002000999,at_least_three,13,198,0.065657
1,2011,002000999,direct,2,198,0.010101
2,2011,002000999,direct_failed_open,133,198,0.671717
3,2011,002000999,open,50,198,0.252525
4,2011,004000995,at_least_three,5,187,0.026738
...,...,...,...,...,...,...
59878,2022,T0O,direct,5,5,1.000000
59879,2022,TOM,direct,11,11,1.000000
59880,2022,TQA,direct,17,17,1.000000
59881,2022,W3S,direct,22,22,1.000000


In [6]:
buyers_direct = buyers[buyers['procedure_type_fixed'] == 'direct'][['contract_year', 'purchasing_unit_id', 'prop_per_procedure']].rename(columns={'prop_per_procedure':'prop_b_direct'})
buyers_failedopen = buyers[buyers['procedure_type_fixed'] == 'direct_failed_open'][['contract_year', 'purchasing_unit_id', 'prop_per_procedure']].rename(columns={'prop_per_procedure':'prop_b_failedopen'})
buyer_direct_full = buyers[buyers['procedure_type_fixed'].isin(['direct', 'direct_failed_open'])]
buyer_direct_full =  buyer_direct_full.groupby(['contract_year', 'purchasing_unit_id'])['prop_per_procedure'].sum().reset_index(name='prop_b_direct_full')

In [7]:
contracts = contracts.merge(buyers_direct, on=['contract_year', 'purchasing_unit_id'], how='left')
contracts = contracts.merge(buyers_failedopen, on=['contract_year', 'purchasing_unit_id'], how='left')
contracts = contracts.merge(buyer_direct_full, on=['contract_year', 'purchasing_unit_id'], how='left')

In [8]:
contracts.columns

Index(['supplier_name_clean', 'purchasing_unit_id', 'contract_price_mx',
       'procedure_type_fixed', 'contract_initial_date', 'contract_year',
       'providers_registry_code', 'prov_id', 'prop_b_direct',
       'prop_b_failedopen', 'prop_b_direct_full'],
      dtype='object')

In [9]:
suppliers = contracts[['contract_year', 'supplier_name_clean', 'procedure_type_fixed']]
suppliers = suppliers.groupby(['contract_year', 'supplier_name_clean', 'procedure_type_fixed']).size().reset_index(name='ncontracts_per_procedure')
suppliers['ncontracts'] = suppliers.groupby(['contract_year', 'supplier_name_clean'])['ncontracts_per_procedure'].transform('sum')
suppliers['prop_per_procedure'] = suppliers['ncontracts_per_procedure'] / suppliers['ncontracts']
suppliers

,contract_year,supplier_name_clean,procedure_type_fixed,ncontracts_per_procedure,ncontracts,prop_per_procedure
0,2011,0donnellriveranataliemarie,direct_failed_open,1,1,1.00
1,2011,10distribucionesespeciales,direct_failed_open,1,1,1.00
2,2011,130ingenieriayarquitecturasadecv,at_least_three,1,1,1.00
3,2011,27micrasinternacional,at_least_three,1,4,0.25
4,2011,27micrasinternacional,direct_failed_open,2,4,0.50
...,...,...,...,...,...,...
778969,2022,zyanyasahiandiazperez,direct_failed_open,1,1,1.00
778970,2022,zyanyaselenecuellarrosales,direct_failed_open,1,1,1.00
778971,2022,zyxasadecv,direct_failed_open,1,1,1.00
778972,2022,zyzlilamargaritagomezarce,direct_failed_open,3,3,1.00


In [10]:
suppliers_direct = suppliers[suppliers['procedure_type_fixed'] == 'direct'][['contract_year', 'supplier_name_clean', 'prop_per_procedure']].rename(columns={'prop_per_procedure':'prop_s_direct'})
suppliers_failedopen = suppliers[suppliers['procedure_type_fixed'] == 'direct_failed_open'][['contract_year', 'supplier_name_clean', 'prop_per_procedure']].rename(columns={'prop_per_procedure':'prop_s_failedopen'})
supplier_direct_full = suppliers[suppliers['procedure_type_fixed'].isin(['direct', 'direct_failed_open'])]
supplier_direct_full =  supplier_direct_full.groupby(['contract_year', 'supplier_name_clean'])['prop_per_procedure'].sum().reset_index(name='prop_s_direct_full')

In [11]:
contracts = contracts.merge(suppliers_direct, on=['contract_year', 'supplier_name_clean'], how='left')
contracts = contracts.merge(suppliers_failedopen, on=['contract_year', 'supplier_name_clean'], how='left')
contracts = contracts.merge(supplier_direct_full, on=['contract_year', 'supplier_name_clean'], how='left')

In [12]:
contracts['prop_b_direct'] = contracts['prop_b_direct'].fillna(0)
contracts['prop_b_failedopen'] = contracts['prop_b_failedopen'].fillna(0)
contracts['prop_b_direct_full'] = contracts['prop_b_direct_full'].fillna(0)
contracts['prop_s_direct'] = contracts['prop_s_direct'].fillna(0)
contracts['prop_s_failedopen'] = contracts['prop_s_failedopen'].fillna(0)
contracts['prop_s_direct_full'] = contracts['prop_s_direct_full'].fillna(0)


### Not in RUPC

In [13]:
# import excel
rupc = pd.read_excel(here() / 'data/contracts_data/RUPC_240519124309.xlsx')
rupc.columns = ['providers_registry_code', 'RFC', 'supplier_name', 'country', 'state', 'size', 'supplier_type', 'sector', 'purpose', 'ncontracts', 'registry_date', 'evaluated_contracts_laassp', 'grade_laassp', 'evaluated_contracts_lopsrm', 'grade_lopsrm', 'website']
#add column suffix _rupc
rupc.columns = [f'{col}_rupc' for col in rupc.columns]
rupc['supplier_name_clean_rupc'] = utils.clean_names(rupc, 'supplier_name_rupc')
rupc = rupc[['supplier_name_clean_rupc', 'providers_registry_code_rupc']]
print(rupc.shape)
rupc.drop_duplicates(inplace=True)
print(rupc.shape)
rupc_dict = rupc.set_index('supplier_name_clean_rupc')['providers_registry_code_rupc'].to_dict()


/mnt/sdb1/marti/miniconda3/envs/det_corr_mex_env/lib/python3.7/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(21649, 2)
(21649, 2)


In [14]:
len(rupc_dict)

21647

In [15]:
c_rupc = contracts.copy()[['supplier_name_clean', 'providers_registry_code']]
print(c_rupc.shape)
c_rupc['providers_registry_code'] = np.where(c_rupc['providers_registry_code'] == -999, np.nan, c_rupc['providers_registry_code'])
c_rupc = c_rupc.dropna().reset_index(drop=True)
print(c_rupc.shape)
c_rupc = c_rupc.drop_duplicates()
print(c_rupc.shape)
c_rupc_dict = c_rupc.set_index('supplier_name_clean')['providers_registry_code'].to_dict()

(2308908, 2)
(836202, 2)
(23062, 2)


In [16]:
contracts['providers_registry_code_new'] = contracts['supplier_name_clean'].map(c_rupc_dict)


In [18]:
contracts['providers_registry_code_rupc'] = contracts['supplier_name_clean'].map(rupc_dict)

In [19]:
contracts['notinRUPC'] = np.where((contracts['providers_registry_code_new'].notnull()) | (contracts['providers_registry_code_rupc'].notnull()) , 0, 1)

In [21]:
contracts.drop(columns=[ 'providers_registry_code_rupc', 'providers_registry_code_new'], inplace=True)

### number of contracts per buyer-supplier-week

I use contract inital date instead of award date since i have no missing data there, so I dont have to impute and more than 75% of the difference between contract initial date and award date is between a week

In [22]:
contracts['week_number'] = contracts['contract_initial_date'].dt.isocalendar().week

In [ ]:
contracts['week_number'] = contracts['week_number'].astype('int64')


In [29]:
print(contracts['contract_year'].dtype)
print(contracts['purchasing_unit_id'].dtype)
print(contracts['supplier_name_clean'].dtype)
print(contracts['week_number'].dtype)

int64
object
object
int64


In [30]:
#ncontractsxbuyerxsupplierxweek
#contracts['ncontracts_bsw'] = contracts.groupby(['contract_year', 'purchasing_unit_id', 'supplier_name_clean', 'week_number']).transform('size')
contracts['ncontracts_bsw'] = (
    contracts
    .groupby(
        ['contract_year', 'purchasing_unit_id', 'supplier_name_clean', 'week_number']
    )['week_number']
    .transform('size')
)

### spending per active week

In [32]:
#active weeks
contracts['active_weeks_bs'] = contracts.groupby(['contract_year', 'purchasing_unit_id', 'supplier_name_clean'])['week_number'].transform('nunique')

In [33]:
#spending per active week
contracts['spending_bs'] = contracts.groupby(['contract_year', 'purchasing_unit_id', 'supplier_name_clean'])['contract_price_mx'].transform('sum')
contracts['spending_bs_aw'] = contracts['spending_bs'] / contracts['active_weeks_bs']

In [34]:
contracts.drop(columns=['spending_bs'], inplace=True)

### number of contracts per buyer-supplier pair

ncontracts_bs is the number of contracts per buyer-supplier pair
ncontracts_bs_odds is ncontracts_bs over the yearly median. Therefore I'm measuring how far is from the median

In [36]:
contracts['ncontracts_bs']  = contracts.groupby(['contract_year', 'purchasing_unit_id', 'supplier_name_clean'])['supplier_name_clean'].transform('size')

In [38]:
contracts['ncontracts_bs'].mean()

74.15207968442225

In [39]:
contracts['ncontracts_bs_yearmedian'] = contracts.groupby('contract_year')['ncontracts_bs'].transform('median')

In [40]:
contracts['ncontracts_bs_odds'] = contracts['ncontracts_bs'] / contracts['ncontracts_bs_yearmedian']


In [41]:
contracts.drop(columns=['ncontracts_bs_yearmedian'], inplace=True)

### fragmented contracts

if contract_price_bsw > (quantile 0.75 of contract_price_b & ncontracts_bsw > 1), then fragmented contract

Only among direct, direct_failed_open and at_least_three

In [42]:
open_median_yearly = contracts.groupby(['contract_year', 'procedure_type_fixed'])['contract_price_mx'].quantile(0.5).reset_index()
open_median_yearly['contract_price_mx'] = np.round(open_median_yearly['contract_price_mx'], 0)
open_median_yearly = open_median_yearly \
    .pivot(index='contract_year', columns='procedure_type_fixed', values='contract_price_mx').sort_values('contract_year').reset_index()


In [43]:
open_median_yearly


procedure_type_fixed,contract_year,at_least_three,direct,direct_failed_open,open,other
0,2011,766380.0,62727.0,69864.0,1021490.0,NaN
1,2012,701403.0,82380.0,75747.0,916470.0,NaN
2,2013,757245.0,81320.0,74000.0,929330.0,NaN
3,2014,850000.0,89153.0,73000.0,1110150.0,7688900.0
4,2015,817628.0,77773.0,69250.0,849538.0,687744.0
5,2016,870697.0,77349.0,65000.0,920847.0,766557.0
6,2017,909119.0,81356.0,62549.0,910575.0,525244.0
7,2018,1077523.0,149773.0,94200.0,833794.0,415380.0
8,2019,315044.0,4663.0,74240.0,916721.0,523872.0
9,2020,484276.0,4595.0,108676.0,982200.0,188598.0


In [44]:
open_median_yearly = open_median_yearly[['contract_year', 'open']]
open_median_yearly

procedure_type_fixed,contract_year,open
0,2011,1021490.0
1,2012,916470.0
2,2013,929330.0
3,2014,1110150.0
4,2015,849538.0
5,2016,920847.0
6,2017,910575.0
7,2018,833794.0
8,2019,916721.0
9,2020,982200.0


In [45]:
var2keep = ['prov_id', 'contract_year', 'supplier_name_clean', 'purchasing_unit_id', 'week_number', 'procedure_type_fixed', 'contract_price_mx', 'ncontracts_bsw']

fragmented_contracts_df = contracts[contracts['procedure_type_fixed'].isin(['direct', 'direct_failed_open', 'at_least_three'])][var2keep]



In [46]:
fragmented_contracts_df

,prov_id,contract_year,supplier_name_clean,purchasing_unit_id,week_number,procedure_type_fixed,contract_price_mx,ncontracts_bsw
0,0,2011,tcaempresarialsadecv,050GYR005,50,direct_failed_open,146343.00,1
1,1,2011,instrumentosyaccesoriosautomatizadossadecv,050GYR005,50,direct_failed_open,6250.00,1
2,2,2011,dacegacorporation,050GYR025,51,direct_failed_open,54400.00,1
3,3,2011,lizbethapariciorazo,016000997,40,direct_failed_open,88579.20,1
4,4,2011,victormiguelvallejojuarez,821045953,38,at_least_three,163970.68,34
...,...,...,...,...,...,...,...,...
2308903,2308903,2022,formaseficientessadecv,QDV,6,direct,364613.67,1
2308904,2308904,2022,cosmopapelsadecv,QDV,7,direct,346016.40,1
2308905,2308905,2022,farvisaninsumosinstitucionalessadecv,QDV,6,direct,1885.35,1
2308906,2308906,2022,formaseficientessadecv,NBD,8,direct,1036815.04,1


In [47]:
fragmented_contracts_df = fragmented_contracts_df.merge(open_median_yearly, on='contract_year', how='left')

In [48]:
fragmented_contracts_df

,prov_id,contract_year,supplier_name_clean,purchasing_unit_id,week_number,procedure_type_fixed,contract_price_mx,ncontracts_bsw,open
0,0,2011,tcaempresarialsadecv,050GYR005,50,direct_failed_open,146343.00,1,1021490.0
1,1,2011,instrumentosyaccesoriosautomatizadossadecv,050GYR005,50,direct_failed_open,6250.00,1,1021490.0
2,2,2011,dacegacorporation,050GYR025,51,direct_failed_open,54400.00,1,1021490.0
3,3,2011,lizbethapariciorazo,016000997,40,direct_failed_open,88579.20,1,1021490.0
4,4,2011,victormiguelvallejojuarez,821045953,38,at_least_three,163970.68,34,1021490.0
...,...,...,...,...,...,...,...,...,...
1957565,2308903,2022,formaseficientessadecv,QDV,6,direct,364613.67,1,1102000.0
1957566,2308904,2022,cosmopapelsadecv,QDV,7,direct,346016.40,1,1102000.0
1957567,2308905,2022,farvisaninsumosinstitucionalessadecv,QDV,6,direct,1885.35,1,1102000.0
1957568,2308906,2022,formaseficientessadecv,NBD,8,direct,1036815.04,1,1102000.0


In [49]:
fragmented_contracts_df['contract_price_bsw'] = fragmented_contracts_df.groupby(['contract_year', 'supplier_name_clean', 'purchasing_unit_id', 'week_number'])['contract_price_mx'].transform('sum')

In [50]:
fragmented_contracts_df['fragmented_contract_odds'] = fragmented_contracts_df['contract_price_bsw'] / fragmented_contracts_df['open']

In [51]:
fragmented_contracts_df['fragmented_contract_odds'].describe()

count    1.957570e+06
mean     2.425724e+00
std      4.448199e+01
min      1.098207e-08
25%      6.003263e-02
50%      2.005656e-01
75%      8.885277e-01
max      2.631792e+04
Name: fragmented_contract_odds, dtype: float64

In [52]:
fragmented_contracts_df.columns

Index(['prov_id', 'contract_year', 'supplier_name_clean', 'purchasing_unit_id',
       'week_number', 'procedure_type_fixed', 'contract_price_mx',
       'ncontracts_bsw', 'open', 'contract_price_bsw',
       'fragmented_contract_odds'],
      dtype='object')

In [53]:
fragmented_contracts_df = fragmented_contracts_df[['prov_id', 'fragmented_contract_odds']]

In [54]:
fragmented_contracts_df.shape

(1957570, 2)

In [55]:
contracts = pd.merge(contracts, fragmented_contracts_df, how='left', on=['prov_id'])

In [56]:
contracts['fragmented_contract_odds'].fillna(0, inplace=True)

In [57]:
contracts.shape

(2308908, 22)

In [58]:
contracts.isnull().sum() / contracts.shape[0]

supplier_name_clean         0.000000
purchasing_unit_id          0.000000
contract_price_mx           0.000000
procedure_type_fixed        0.002651
contract_initial_date       0.000000
contract_year               0.000000
providers_registry_code     0.000000
prov_id                     0.000000
prop_b_direct               0.000000
prop_b_failedopen           0.000000
prop_b_direct_full          0.000000
prop_s_direct               0.000000
prop_s_failedopen           0.000000
prop_s_direct_full          0.000000
notinRUPC                   0.000000
week_number                 0.000000
ncontracts_bsw              0.000000
active_weeks_bs             0.000000
spending_bs_aw              0.000000
ncontracts_bs               0.000000
ncontracts_bs_odds          0.000000
fragmented_contract_odds    0.000000
dtype: float64

In [59]:
contracts.drop(columns=['week_number'], inplace=True)

### Contract concentration

In [60]:
#count contracts per buyer
contracts['contracts_per_buyer'] = contracts.groupby(['contract_year', 'purchasing_unit_id'])['purchasing_unit_id'].transform('size')

In [61]:
contracts['contracts_per_buyer'].describe()

count    2.308908e+06
mean     3.524793e+03
std      7.803555e+03
min      1.000000e+00
25%      1.300000e+02
50%      4.710000e+02
75%      1.575000e+03
max      3.113500e+04
Name: contracts_per_buyer, dtype: float64

In [62]:
#count contracts per buyer and supplier
contracts['contracts_per_buyer_supplier'] = contracts.groupby(['contract_year', 'purchasing_unit_id', 'supplier_name_clean'])['supplier_name_clean'].transform('size')

In [63]:
contracts['contracts_per_buyer_supplier'].mean()

74.15207968442225

In [64]:
contracts['prop_contracts_bs'] = contracts['contracts_per_buyer_supplier'] / contracts['contracts_per_buyer']

In [65]:
contracts['prop_contracts_bs'].describe()

count    2.308908e+06
mean     3.178008e-02
std      7.483391e-02
min      3.211819e-05
25%      4.137931e-03
50%      1.145475e-02
75%      3.030303e-02
max      1.000000e+00
Name: prop_contracts_bs, dtype: float64

In [66]:
#drop variables
contracts.drop(columns=['contracts_per_buyer', 'contracts_per_buyer_supplier'], inplace=True)

In [67]:
contracts.isnull().sum() / contracts.shape[0]

supplier_name_clean         0.000000
purchasing_unit_id          0.000000
contract_price_mx           0.000000
procedure_type_fixed        0.002651
contract_initial_date       0.000000
contract_year               0.000000
providers_registry_code     0.000000
prov_id                     0.000000
prop_b_direct               0.000000
prop_b_failedopen           0.000000
prop_b_direct_full          0.000000
prop_s_direct               0.000000
prop_s_failedopen           0.000000
prop_s_direct_full          0.000000
notinRUPC                   0.000000
ncontracts_bsw              0.000000
active_weeks_bs             0.000000
spending_bs_aw              0.000000
ncontracts_bs               0.000000
ncontracts_bs_odds          0.000000
fragmented_contract_odds    0.000000
prop_contracts_bs           0.000000
dtype: float64

In [68]:
#drop variables
contracts.drop(columns=['providers_registry_code', 'procedure_type_fixed', 'contract_price_mx', 'contract_initial_date'], inplace=True)

In [69]:
contracts.isnull().sum() / contracts.shape[0]

supplier_name_clean         0.0
purchasing_unit_id          0.0
contract_year               0.0
prov_id                     0.0
prop_b_direct               0.0
prop_b_failedopen           0.0
prop_b_direct_full          0.0
prop_s_direct               0.0
prop_s_failedopen           0.0
prop_s_direct_full          0.0
notinRUPC                   0.0
ncontracts_bsw              0.0
active_weeks_bs             0.0
spending_bs_aw              0.0
ncontracts_bs               0.0
ncontracts_bs_odds          0.0
fragmented_contract_odds    0.0
prop_contracts_bs           0.0
dtype: float64

In [70]:
#save as feather
contracts.to_feather(processed_data / 'imco.feather')